# 01 · Register rainfall: the native 1-minute release and its 5-minute analysis derivation

**Depends on 00.** The published CML rainfall product is **native 1-minute**
(`CZ_CML_RAINRATE_GRAF_ITU2005_PASTOREK_1MIN_20210401_20211031.nc`). The original Graf wet/dry → constant baseline → Pastorek WAA → ITU-R P.838-3 (2005) retrieval remains unchanged in `original/`. This notebook registers the release and materializes the **analysis 5-minute series**, which are an exact function of the released 1-minute values: a directional 5-minute rate exists only where all five 1-minute samples of the block are valid, the wet flag is set when any of those valid minutes was wet, and the physical-link mean is taken over the finite directional 5-minute rates afterwards. This derivation was verified bit for bit against the historical validated 5-minute reference for all 2,219 links (`docs/EVIDENCE.md`). To rerun the legacy full-history 5-minute retrieval instead, set `RECOMPUTE=True`; this is expensive and uses the entire QC input timeline to preserve the Q80/baseline history. The next notebook automatically loads the path registered here.

Changing scientific parameters would define a new experiment; the original publication producer intentionally uses its documented parameter constants. Never re-label a modified method as manuscript reproduction.

**About this notebook.** Needs: the released native 1-minute rainfall file (and the preprocessed CML file only for the optional recomputation). Parameters: `RECOMPUTE` (off by default), `WORKERS`. Makes: registers the released 1-minute product and derives the analysis 5-minute rainfall consumed by the next notebooks; with `RECOMPUTE=True` it reruns the full original retrieval. Cost: about a minute for the one-time 5-minute derivation, hours for the full 2,219 link recomputation. Fidelity: exact; the 5-minute derivation reproduced the validated reference bit for bit on all links, and the full recomputation reproduced that reference in every value and mask. Details: `docs/NOTEBOOK_GUIDE.md`.

In [4]:
from pathlib import Path
import sys, json
# One shared configuration for ALL notebooks (no per-notebook path editing).
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'tools_verify_originals.py').is_file(), f'Wrong notebook working folder: {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
from essd_cz.connected import Project
SETTINGS=json.loads((ROOT/'config/active_profile.json').read_text(encoding='utf-8'))
PROFILE=SETTINGS.get('profile','hpc')
if PROFILE not in ('hpc','user'):raise ValueError("Profile must be 'hpc' or 'user'")
CONFIG=ROOT/('config/hpc_source.example.json' if PROFILE=='hpc' else 'config/user_paths.json')
WORKDIR=ROOT/SETTINGS.get('workspace','results/connected_workflow')
project=Project(CONFIG,WORKDIR)
print('Profile:',PROFILE,'Data config:',project.config,'Output:',project.output)

import inspect
from essd_cz.connected import Project

print(inspect.signature(Project.recompute_rainfall))

Profile: hpc Data config: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/config/hpc_source.example.json Output: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow
(self, workers=4, python_executable=None)


In [5]:
from pathlib import Path
import sys, json
# One shared configuration for ALL notebooks (no per-notebook path editing).
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'tools_verify_originals.py').is_file(), f'Wrong notebook working folder: {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
from essd_cz.connected import Project
SETTINGS=json.loads((ROOT/'config/active_profile.json').read_text(encoding='utf-8'))
PROFILE=SETTINGS.get('profile','hpc')
if PROFILE not in ('hpc','user'):raise ValueError("Profile must be 'hpc' or 'user'")
CONFIG=ROOT/('config/hpc_source.example.json' if PROFILE=='hpc' else 'config/user_paths.json')
WORKDIR=ROOT/SETTINGS.get('workspace','results/connected_workflow')
project=Project(CONFIG,WORKDIR)
print('Profile:',PROFILE,'Data config:',project.config,'Output:',project.output)


Profile: hpc Data config: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/config/hpc_source.example.json Output: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow


In [ ]:
project.get('exploration_complete')  # requires notebook 00
RECOMPUTE = False      # False: use the released rainfall without recomputing; safest first run
WORKERS = 4             # used only when RECOMPUTE is True
# One shared path for the scientific Python, per active profile. External users
# leave 'user' at null to use their notebook kernel, or set their own Python.
SCIENCE_PYTHON = SETTINGS.get('science_python_by_profile', {}).get(PROFILE)
print('Scientific Python:', SCIENCE_PYTHON or sys.executable)
if RECOMPUTE:
    # Expensive legacy verification: recomputes the 5-minute reference for all
    # 2,219 links over the ENTIRE available QC history (hpc profile sources).
    rainfall_file = project.recompute_rainfall(
        workers=WORKERS, python_executable=SCIENCE_PYTHON)
else:
    # Registers the released NATIVE 1-MINUTE product and materializes its exact
    # analysis 5-minute derivation once per workspace (docs/03_RAINFALL_EXPLAINED.md).
    rainfall_file = project.source_rainfall()
print('Next notebook receives:',rainfall_file)

In [ ]:
import xarray as xr
with xr.open_dataset(rainfall_file) as ds:
    print('Registered analysis rain dimensions:',dict(ds.sizes))
    print('Registered analysis rain variables:',list(ds.data_vars))
print('The released product is the NATIVE 1-MINUTE rainfall (mm/h). The file')
print('registered above carries the analysis 5-minute series derived from it:')
print('mean of five complete valid 1-minute directional samples, wet if any')
print('valid wet minute, physical mean over finite directional 5-minute rates.')
print('No units have been converted or altered.')